# Data Analysis Notebook

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" # type: ignore

## Import Paths

In [3]:
from pathlib import Path
from utils.path_loader import load_paths
paths = load_paths()

PROJECT_ROOT = paths["project_root"]
# print(PROJECT_ROOT)

## Import Functions

In [ ]:

from train_yolo.functions.yolo_results import collect_models_results
from train_yolo.functions.yolo_speed import collect_models_speed

In [ ]:
# Se não está a a fazer reload por alguma razão
import importlib
import train_yolo.functions.yolo_results as results
import train_yolo.functions.yolo_speed as speed

# força reload real
importlib.reload(results)  
results.collect_models_results
importlib.reload(speed) 
speed.collect_models_speed


<module 'train_yolo.functions.yolo_results' from 'C:\\Users\\danie\\Desktop\\My_Things\\Hobbies\\VSCODE\\Virtual Envorinments\\CapStone\\src\\train_yolo\\functions\\yolo_results.py'>

<function train_yolo.functions.yolo_results.collect_models_results(models_root: pathlib.Path = WindowsPath('C:/Users/danie/Desktop/My_Things/Hobbies/VSCODE/Virtual Envorinments/CapStone/src/train_yolo'))>

<module 'train_yolo.functions.yolo_speed' from 'C:\\Users\\danie\\Desktop\\My_Things\\Hobbies\\VSCODE\\Virtual Envorinments\\CapStone\\src\\train_yolo\\functions\\yolo_speed.py'>

<function train_yolo.functions.yolo_speed.collect_models_speed(models_root: pathlib.Path = WindowsPath('C:/Users/danie/Desktop/My_Things/Hobbies/VSCODE/Virtual Envorinments/CapStone/src/train_yolo'), num_images: int = 150, device: str = 'cpu')>

## Load the Data

In [ ]:
df_yolo_results = collect_models_results()

print("Número de trains encontrados:", len(df_yolo_results))
df_yolo_results.head()


Número de trains encontrados: 22


,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
0,yolo11m,text-photo,train-150e8b,150,8,137,0.96288,0.88083,0.93996,0.94068,0.63244,0.40604,0.94324,True,src\train_yolo\yolo11m-text-photo\train-150e8b
1,yolo11m,text-photo,train-150e4b,150,4,140,0.95996,0.87510,0.93518,0.94326,0.63579,0.38550,0.93975,True,src\train_yolo\yolo11m-text-photo\train-150e4b
2,yolo11s,text-photo,train-150e2b,150,2,130,0.96380,0.87301,0.93939,0.94562,0.61990,0.42465,0.91354,True,src\train_yolo\yolo11s-text-photo\train-150e2b
3,yolo11s,text-photo,train-150e6b,150,6,133,0.95890,0.87256,0.92871,0.94878,0.63772,0.42279,0.92648,True,src\train_yolo\yolo11s-text-photo\train-150e6b
4,yolo11s,text-photo,train-150e14b,150,14,114,0.96047,0.87143,0.94053,0.93488,0.64074,0.41563,0.92860,True,src\train_yolo\yolo11s-text-photo\train-150e14b


In [17]:
df_yolo_speed = collect_models_speed(device=0, num_images=150)

print("Número de trains encontrados:", len(df_yolo_speed))
df_yolo_speed.head()



=== YOLO SPEED BENCHMARK ===
Project root: C:\Users\danie\Desktop\My_Things\Hobbies\VSCODE\Virtual Envorinments\CapStone
Device: 0
Images per test: 150

Número de trains encontrados: 23

[1/23] Benchmarking best.pt | task=text-photo | train=train-150e4b
    → 64.19 ms/img | 15.58 FPS

[2/23] Benchmarking best.pt | task=text-photo | train=train-150e8b
    → 65.02 ms/img | 15.38 FPS

[3/23] Benchmarking best.pt | task=skill_table | train=train-150e2b
    → 43.01 ms/img | 23.25 FPS

[4/23] Benchmarking best.pt | task=skill_table | train=train-150e4b
    → 37.01 ms/img | 27.02 FPS

[5/23] Benchmarking best.pt | task=skill_table | train=train-150e8b
    → 33.32 ms/img | 30.01 FPS

[6/23] Benchmarking best.pt | task=text-photo | train=train-150e10b
    → 38.53 ms/img | 25.95 FPS

[7/23] Benchmarking best.pt | task=text-photo | train=train-150e11b
    → 39.53 ms/img | 25.30 FPS

[8/23] Benchmarking best.pt | task=text-photo | train=train-150e12b
    → 35.77 ms/img | 27.96 FPS

[9/23] Benchma

,yolo_version,task,train_run,ms_per_image,fps,num_images,device,model_path
4,yolo11s,skill_table,train-150e8b,33.318424,30.013425,144,0,src\train_yolo\yolo11s-skill_table\train-150e8...
12,yolo11s,text-photo,train-150e3b,33.701020,29.672693,134,0,src\train_yolo\yolo11s-text-photo\train-150e3b...
13,yolo11s,text-photo,train-150e4b,34.270787,29.179371,134,0,src\train_yolo\yolo11s-text-photo\train-150e4b...
9,yolo11s,text-photo,train-150e16b,34.894363,28.657924,134,0,src\train_yolo\yolo11s-text-photo\train-150e16...
14,yolo11s,text-photo,train-150e5b,35.226612,28.387629,134,0,src\train_yolo\yolo11s-text-photo\train-150e5b...


In [23]:
def merge_yolo_results_and_speed(df_yolo_results: pd.DataFrame,
                                 df_yolo_speed: pd.DataFrame) -> pd.DataFrame:
    """
    Junta métricas de treino (results) com métricas de velocidade (speed)
    usando as chaves:
        - yolo_version
        - task
        - train_run

    Retorna DataFrame combinado.
    """

    if df_yolo_results.empty:
        raise ValueError("df_yolo_results está vazio")

    if df_yolo_speed.empty:
        raise ValueError("df_yolo_speed está vazio")

    required_cols = {"yolo_version", "task", "train_run"}

    if not required_cols.issubset(df_yolo_results.columns):
        raise ValueError("df_yolo_results não contém as colunas necessárias")

    if not required_cols.issubset(df_yolo_speed.columns):
        raise ValueError("df_yolo_speed não contém as colunas necessárias")

    # Merge interno apenas nas runs que existem nos dois
    df_merged = pd.merge(
        df_yolo_results,
        df_yolo_speed,
        on=["yolo_version", "task", "train_run"],
        how="inner",
        suffixes=("", "_speed"),
    )

    # Ordenação útil para análise científica
    if "mAP50-95" in df_merged.columns:
        df_merged = df_merged.sort_values(
            by=["task", "mAP50-95", "fps"],
            ascending=[True, False, False],
        ).reset_index(drop=True)

    return df_merged

from pathlib import Path
import pandas as pd


def save_dataframe(
    df: pd.DataFrame,
    output_path: Path | str,
    filename: str = "df_yolo_final",
    format: str = "csv",
) -> Path:
    """
    Guarda um DataFrame no caminho escolhido.

    Args:
        df: DataFrame a guardar
        output_path: pasta de destino
        filename: nome do ficheiro (sem extensão)
        format: 'csv' ou 'parquet'

    Returns:
        Caminho completo do ficheiro guardado.
    """

    if df.empty:
        raise ValueError("O DataFrame está vazio, nada para guardar.")

    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    format = format.lower()

    if format == "csv":
        file_path = output_path / f"{filename}.csv"
        df.to_csv(file_path, index=False)

    elif format == "parquet":
        file_path = output_path / f"{filename}.parquet"
        df.to_parquet(file_path, index=False)

    else:
        raise ValueError("Formato inválido. Usa 'csv' ou 'parquet'.")

    print(f"[SAVE] DataFrame guardado em: {file_path}")

    return file_path



In [21]:
df_yolo_final = merge_yolo_results_and_speed(df_yolo_results, df_yolo_speed)

df_yolo_final.head()

,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path,ms_per_image,fps,num_images,device,model_path
0,yolo11s,skill_table,train-150e8b,150,8,146,0.97745,0.85591,0.97727,0.96314,0.59987,0.40400,0.78381,True,src\train_yolo\yolo11s-skill_table\train-150e8b,33.318424,30.013425,144,0,src\train_yolo\yolo11s-skill_table\train-150e8...
1,yolo11s,skill_table,train-150e4b,150,4,149,0.98565,0.84150,0.96532,0.96069,0.58958,0.41314,0.74897,True,src\train_yolo\yolo11s-skill_table\train-150e4b,37.011741,27.018453,144,0,src\train_yolo\yolo11s-skill_table\train-150e4...
2,yolo11m,text-photo,train-150e8b,150,8,137,0.96288,0.88083,0.93996,0.94068,0.63244,0.40604,0.94324,True,src\train_yolo\yolo11m-text-photo\train-150e8b,65.015111,15.381040,134,0,src\train_yolo\yolo11m-text-photo\train-150e8b...
3,yolo11m,text-photo,train-150e4b,150,4,140,0.95996,0.87510,0.93518,0.94326,0.63579,0.38550,0.93975,True,src\train_yolo\yolo11m-text-photo\train-150e4b,64.194861,15.577571,134,0,src\train_yolo\yolo11m-text-photo\train-150e4b...
4,yolo11s,text-photo,train-150e2b,150,2,130,0.96380,0.87301,0.93939,0.94562,0.61990,0.42465,0.91354,True,src\train_yolo\yolo11s-text-photo\train-150e2b,35.568214,28.114990,134,0,src\train_yolo\yolo11s-text-photo\train-150e2b...


In [22]:
output_dir = PROJECT_ROOT / "src" / "train_yolo" 

save_dataframe(
    df_yolo_final,
    output_path=output_dir,
    filename="yolo_models_benchmarks",
    format="csv",
)


[SAVE] DataFrame guardado em: C:\Users\danie\Desktop\My_Things\Hobbies\VSCODE\Virtual Envorinments\CapStone\src\train_yolo\yolo_models_benchmarks.csv


WindowsPath('C:/Users/danie/Desktop/My_Things/Hobbies/VSCODE/Virtual Envorinments/CapStone/src/train_yolo/yolo_models_benchmarks.csv')

## 1. Yolo Models

### 1.1 Explore the Data

#### 1.1.1 Ordenar o dataset

In [ ]:
df_yolo_results_sorted = df_yolo_results.sort_values("mAP50-95", ascending=False)

df_yolo_results_sorted

,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
0,yolo11m,text-photo,train-150e8b,150,8,137,0.96288,0.88083,0.93996,0.94068,0.63244,0.40604,0.94324,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
1,yolo11m,text-photo,train-150e4b,150,4,140,0.95996,0.87510,0.93518,0.94326,0.63579,0.38550,0.93975,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
2,yolo11s,text-photo,train-150e2b,150,2,130,0.96380,0.87301,0.93939,0.94562,0.61990,0.42465,0.91354,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
3,yolo11s,text-photo,train-150e6b,150,6,133,0.95890,0.87256,0.92871,0.94878,0.63772,0.42279,0.92648,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
4,yolo11s,text-photo,train-150e14b,150,14,114,0.96047,0.87143,0.94053,0.93488,0.64074,0.41563,0.92860,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
5,yolo11s,text-photo,train-150e5b,150,5,117,0.96036,0.87128,0.93106,0.93994,0.63134,0.42985,0.91753,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
6,yolo11s,text-photo,train-150e4b,150,4,135,0.95843,0.87079,0.93390,0.95063,0.62740,0.40453,0.91603,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
7,yolo11s,text-photo,train-150e3b,150,3,135,0.95861,0.86808,0.93967,0.94819,0.62629,0.40905,0.91482,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
8,yolo11s,text-photo,train-150e12b,150,12,130,0.95778,0.86745,0.92933,0.94657,0.64624,0.41473,0.92925,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
9,yolo11s,text-photo,train-150e11b,150,11,117,0.95745,0.86728,0.94080,0.93914,0.64704,0.41271,0.93839,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...


#### 1.1.2 Dataset apenas para cada task:

In [ ]:
# Text-photo
df_text_photo = df_yolo_results[df_yolo_results["task"] == "text-photo"].copy()

print("Runs text-photo:", len(df_text_photo))
df_text_photo

Runs text-photo: 20


,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
0,yolo11m,text-photo,train-150e8b,150,8,137,0.96288,0.88083,0.93996,0.94068,0.63244,0.40604,0.94324,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
1,yolo11m,text-photo,train-150e4b,150,4,140,0.95996,0.87510,0.93518,0.94326,0.63579,0.38550,0.93975,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
2,yolo11s,text-photo,train-150e2b,150,2,130,0.96380,0.87301,0.93939,0.94562,0.61990,0.42465,0.91354,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
3,yolo11s,text-photo,train-150e6b,150,6,133,0.95890,0.87256,0.92871,0.94878,0.63772,0.42279,0.92648,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
4,yolo11s,text-photo,train-150e14b,150,14,114,0.96047,0.87143,0.94053,0.93488,0.64074,0.41563,0.92860,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
5,yolo11s,text-photo,train-150e5b,150,5,117,0.96036,0.87128,0.93106,0.93994,0.63134,0.42985,0.91753,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
6,yolo11s,text-photo,train-150e4b,150,4,135,0.95843,0.87079,0.93390,0.95063,0.62740,0.40453,0.91603,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
7,yolo11s,text-photo,train-150e3b,150,3,135,0.95861,0.86808,0.93967,0.94819,0.62629,0.40905,0.91482,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
8,yolo11s,text-photo,train-150e12b,150,12,130,0.95778,0.86745,0.92933,0.94657,0.64624,0.41473,0.92925,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
9,yolo11s,text-photo,train-150e11b,150,11,117,0.95745,0.86728,0.94080,0.93914,0.64704,0.41271,0.93839,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...


In [ ]:
# Skill_table
df_skill_table = df_yolo_results[df_yolo_results["task"] == "skill_table"].copy()

print("Runs skill_table:", len(df_skill_table))
df_skill_table

Runs skill_table: 1


,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
20,yolo11s,skill_table,train-150e4b,150,4,76,0.9714,0.80958,0.94817,0.94103,0.64413,0.47965,0.76971,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...


#### 1.1.3 Dataset comparativo por batch
* task = text-photo
* batch = 4 e 8

In [11]:
df_text_photo_b4_b8 = df_text_photo[(df_text_photo["batch_config"].isin([4, 8]))].copy()

df_text_photo_b4_b8

,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
0,yolo11m,text-photo,train-150e8b,150,8,137,0.96288,0.88083,0.93996,0.94068,0.63244,0.40604,0.94324,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
1,yolo11m,text-photo,train-150e4b,150,4,140,0.95996,0.87510,0.93518,0.94326,0.63579,0.38550,0.93975,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
6,yolo11s,text-photo,train-150e4b,150,4,135,0.95843,0.87079,0.93390,0.95063,0.62740,0.40453,0.91603,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
10,yolo26s,text-photo,train-150e4b,150,4,98,0.95998,0.86721,0.91895,0.94110,0.65562,0.48420,0.00604,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
11,yolo26s,text-photo,train-150e8b,150,8,111,0.95758,0.86676,0.93220,0.93017,0.66487,0.45587,0.00621,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
12,yolo11s,text-photo,train-150e8b,150,8,69,0.96007,0.86504,0.92560,0.94386,0.62551,0.44502,0.91580,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
15,yolov8s,text-photo,train-150e4b,150,4,113,0.95552,0.86371,0.92931,0.93589,0.62814,0.43453,0.92278,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
19,yolov8s,text-photo,train-150e8b,150,8,93,0.95999,0.86062,0.92113,0.93869,0.65075,0.45245,0.93759,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...


#### 1.1.4 Dataset apenas para text-photo e yolo11s

In [12]:
df_text_photo_yolo11s = df_text_photo[df_models["yolo_version"] == "yolo11s"].copy()

df_text_photo_yolo11s

C:\Users\danie\AppData\Local\Temp\ipykernel_22872\4140887124.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_text_photo_yolo11s = df_text_photo[df_models["yolo_version"] == "yolo11s"].copy()


,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
2,yolo11s,text-photo,train-150e2b,150,2,130,0.96380,0.87301,0.93939,0.94562,0.61990,0.42465,0.91354,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
3,yolo11s,text-photo,train-150e6b,150,6,133,0.95890,0.87256,0.92871,0.94878,0.63772,0.42279,0.92648,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
4,yolo11s,text-photo,train-150e14b,150,14,114,0.96047,0.87143,0.94053,0.93488,0.64074,0.41563,0.92860,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
5,yolo11s,text-photo,train-150e5b,150,5,117,0.96036,0.87128,0.93106,0.93994,0.63134,0.42985,0.91753,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
6,yolo11s,text-photo,train-150e4b,150,4,135,0.95843,0.87079,0.93390,0.95063,0.62740,0.40453,0.91603,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
7,yolo11s,text-photo,train-150e3b,150,3,135,0.95861,0.86808,0.93967,0.94819,0.62629,0.40905,0.91482,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
8,yolo11s,text-photo,train-150e12b,150,12,130,0.95778,0.86745,0.92933,0.94657,0.64624,0.41473,0.92925,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
9,yolo11s,text-photo,train-150e11b,150,11,117,0.95745,0.86728,0.94080,0.93914,0.64704,0.41271,0.93839,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
12,yolo11s,text-photo,train-150e8b,150,8,69,0.96007,0.86504,0.92560,0.94386,0.62551,0.44502,0.91580,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
13,yolo11s,text-photo,train-150e7b,150,7,122,0.95383,0.86491,0.92014,0.95578,0.64016,0.42323,0.92433,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...


#### 1.1.5 Ordenar cada sub-dataset

In [13]:
df_text_photo_sorted = df_text_photo.sort_values("mAP50-95", ascending=False)
df_skill_table_sorted = df_skill_table.sort_values("mAP50-95", ascending=False)
df_text_photo_b4_b8_sorted = df_text_photo_b4_b8.sort_values("mAP50-95", ascending=False)
df_text_photo_yolo11s_sorted = df_text_photo_yolo11s.sort_values("mAP50-95", ascending=False)

In [14]:
df_text_photo_sorted

,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
0,yolo11m,text-photo,train-150e8b,150,8,137,0.96288,0.88083,0.93996,0.94068,0.63244,0.40604,0.94324,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
1,yolo11m,text-photo,train-150e4b,150,4,140,0.95996,0.87510,0.93518,0.94326,0.63579,0.38550,0.93975,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
2,yolo11s,text-photo,train-150e2b,150,2,130,0.96380,0.87301,0.93939,0.94562,0.61990,0.42465,0.91354,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
3,yolo11s,text-photo,train-150e6b,150,6,133,0.95890,0.87256,0.92871,0.94878,0.63772,0.42279,0.92648,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
4,yolo11s,text-photo,train-150e14b,150,14,114,0.96047,0.87143,0.94053,0.93488,0.64074,0.41563,0.92860,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
5,yolo11s,text-photo,train-150e5b,150,5,117,0.96036,0.87128,0.93106,0.93994,0.63134,0.42985,0.91753,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
6,yolo11s,text-photo,train-150e4b,150,4,135,0.95843,0.87079,0.93390,0.95063,0.62740,0.40453,0.91603,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
7,yolo11s,text-photo,train-150e3b,150,3,135,0.95861,0.86808,0.93967,0.94819,0.62629,0.40905,0.91482,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
8,yolo11s,text-photo,train-150e12b,150,12,130,0.95778,0.86745,0.92933,0.94657,0.64624,0.41473,0.92925,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
9,yolo11s,text-photo,train-150e11b,150,11,117,0.95745,0.86728,0.94080,0.93914,0.64704,0.41271,0.93839,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...


In [15]:
df_skill_table_sorted

,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
20,yolo11s,skill_table,train-150e4b,150,4,76,0.9714,0.80958,0.94817,0.94103,0.64413,0.47965,0.76971,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...


In [16]:
df_text_photo_b4_b8_sorted

,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
0,yolo11m,text-photo,train-150e8b,150,8,137,0.96288,0.88083,0.93996,0.94068,0.63244,0.40604,0.94324,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
1,yolo11m,text-photo,train-150e4b,150,4,140,0.95996,0.87510,0.93518,0.94326,0.63579,0.38550,0.93975,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
6,yolo11s,text-photo,train-150e4b,150,4,135,0.95843,0.87079,0.93390,0.95063,0.62740,0.40453,0.91603,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
10,yolo26s,text-photo,train-150e4b,150,4,98,0.95998,0.86721,0.91895,0.94110,0.65562,0.48420,0.00604,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
11,yolo26s,text-photo,train-150e8b,150,8,111,0.95758,0.86676,0.93220,0.93017,0.66487,0.45587,0.00621,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
12,yolo11s,text-photo,train-150e8b,150,8,69,0.96007,0.86504,0.92560,0.94386,0.62551,0.44502,0.91580,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
15,yolov8s,text-photo,train-150e4b,150,4,113,0.95552,0.86371,0.92931,0.93589,0.62814,0.43453,0.92278,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
19,yolov8s,text-photo,train-150e8b,150,8,93,0.95999,0.86062,0.92113,0.93869,0.65075,0.45245,0.93759,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...


In [17]:
df_text_photo_yolo11s_sorted

,yolo_version,task,train_run,epochs_config,batch_config,best_epoch,mAP50,mAP50-95,precision,recall,val_box_loss,val_cls_loss,val_dfl_loss,has_best_weights,run_path
2,yolo11s,text-photo,train-150e2b,150,2,130,0.96380,0.87301,0.93939,0.94562,0.61990,0.42465,0.91354,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
3,yolo11s,text-photo,train-150e6b,150,6,133,0.95890,0.87256,0.92871,0.94878,0.63772,0.42279,0.92648,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
4,yolo11s,text-photo,train-150e14b,150,14,114,0.96047,0.87143,0.94053,0.93488,0.64074,0.41563,0.92860,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
5,yolo11s,text-photo,train-150e5b,150,5,117,0.96036,0.87128,0.93106,0.93994,0.63134,0.42985,0.91753,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
6,yolo11s,text-photo,train-150e4b,150,4,135,0.95843,0.87079,0.93390,0.95063,0.62740,0.40453,0.91603,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
7,yolo11s,text-photo,train-150e3b,150,3,135,0.95861,0.86808,0.93967,0.94819,0.62629,0.40905,0.91482,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
8,yolo11s,text-photo,train-150e12b,150,12,130,0.95778,0.86745,0.92933,0.94657,0.64624,0.41473,0.92925,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
9,yolo11s,text-photo,train-150e11b,150,11,117,0.95745,0.86728,0.94080,0.93914,0.64704,0.41271,0.93839,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
12,yolo11s,text-photo,train-150e8b,150,8,69,0.96007,0.86504,0.92560,0.94386,0.62551,0.44502,0.91580,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...
13,yolo11s,text-photo,train-150e7b,150,7,122,0.95383,0.86491,0.92014,0.95578,0.64016,0.42323,0.92433,True,C:\Users\danie\Desktop\My_Things\Hobbies\VSCOD...


#### 1.1.6 Melhor modelo por sub-dataset

In [18]:
def print_best_model(title: str, row):
    print("=" * 60)
    print(title)
    print("-" * 60)
    print(f"YOLO version : {row['yolo_version']}")
    print(f"Task         : {row['task']}")
    print(f"Train run    : {row['train_run']}")
    print(f"Epochs cfg   : {row['epochs_config']}")
    print(f"Batch cfg    : {row['batch_config']}")
    print(f"Best epoch   : {row['best_epoch']}")
    print()
    print(f"mAP50        : {row['mAP50']:.4f}")
    print(f"mAP50-95     : {row['mAP50-95']:.4f}")
    print(f"Precision    : {row['precision']:.4f}")
    print(f"Recall       : {row['recall']:.4f}")
    print("=" * 60)
    print()


In [20]:
best_text_photo = df_text_photo_sorted.iloc[0]
best_text_photo_b4_b8 = df_text_photo_b4_b8_sorted.iloc[0]
best_text_photo_yolo11s = df_text_photo_yolo11s_sorted.iloc[0]
best_skill_table = df_skill_table_sorted.iloc[0]

In [25]:
print_best_model("BEST MODEL — TEXT-PHOTO (GLOBAL)", best_text_photo)
print_best_model("BEST MODEL — TEXT-PHOTO (BATCH 4 vs 8)", best_text_photo_b4_b8)
print_best_model("BEST MODEL — TEXT-PHOTO (YOLO11s)", best_text_photo_yolo11s)
print_best_model("BEST MODEL — SKILL TABLE", best_skill_table)

BEST MODEL — TEXT-PHOTO (GLOBAL)
------------------------------------------------------------
YOLO version : yolo11m
Task         : text-photo
Train run    : train-150e8b
Epochs cfg   : 150
Batch cfg    : 8
Best epoch   : 137

mAP50        : 0.9629
mAP50-95     : 0.8808
Precision    : 0.9400
Recall       : 0.9407

BEST MODEL — TEXT-PHOTO (BATCH 4 vs 8)
------------------------------------------------------------
YOLO version : yolo11m
Task         : text-photo
Train run    : train-150e8b
Epochs cfg   : 150
Batch cfg    : 8
Best epoch   : 137

mAP50        : 0.9629
mAP50-95     : 0.8808
Precision    : 0.9400
Recall       : 0.9407

BEST MODEL — TEXT-PHOTO (YOLO11s)
------------------------------------------------------------
YOLO version : yolo11s
Task         : text-photo
Train run    : train-150e2b
Epochs cfg   : 150
Batch cfg    : 2
Best epoch   : 130

mAP50        : 0.9638
mAP50-95     : 0.8730
Precision    : 0.9394
Recall       : 0.9456

BEST MODEL — SKILL TABLE
--------------------

#### 1.1.6 Média por versão YOLO apenas para text-photo e batch 4 e 8

In [26]:
df_text_photo_b4_b8.groupby("yolo_version")["mAP50-95"].mean().sort_values(ascending=False)

yolo_version
yolo11m    0.877965
yolo11s    0.867915
yolo26s    0.866985
yolov8s    0.862165
Name: mAP50-95, dtype: float64

### 1.2 Graphics